## Query Generation

### Basic Stats for Basic Cards

In [9]:
import random

In [2]:
card_info_file_path = "data/structured_outputs"

In [3]:
stats_query = "What are the stats for {card}?"

In [38]:
import os

files = os.listdir(card_info_file_path)

In [39]:
files

['bosses',
 'clunkers',
 'companions',
 'enemies',
 'enemy_clunkers',
 'items',
 'minibosses',
 'pets',
 'shades']

In [40]:
queries = []
for file in files:
    
    cards = os.listdir(f'{card_info_file_path}/{file}')
    k = random.randint(2, 3)
    
    random_sample = random.sample(cards, k=k)
    random_sample = [c.split('.html')[0] for c in random_sample]
    print(random_sample)

    for r_s in random_sample:

        queries.append(stats_query.format(card=r_s))

['Frost Bomber', 'Truffle']
['Gachapomper', 'Shroominator']
['Folby', 'Vesta', 'Lupa']
['Grizzle', 'Pecan']
['Mimik', 'Ice Forge', 'Mega Mimik']
["Hongo's Hammer", 'Bom Barrel']
['King Moko', 'The Snow Knight', 'Lumako']
["Lil' Gazi", 'Snoof', 'Loki']
['Pom', 'Chikagoru', 'Beepop']


In [41]:
len(queries)

23

### Tribes

In [ ]:
tribe_query = "What tribe does {card} belong to?"

In [ ]:
"What cards can belong to any tribe?"


In [ ]:
tribe_exclusive_query = "What cards are exclusive to the {tribe_name} tribe?"

In [ ]:
tribe_exclusive_item_query = "What items are exclusive to the {tribe_name} tribe?"

### Abilities

In [ ]:
ability_query = "What ability does {card} have?"

### Relational Questions

In [ ]:
"What cards have only health and counter stats?"
"What cards have only scrap and counter stats?"
"What cards have only scrap and attack stats?"
"What cards have only health, counter, and attack stats?"
"What cards have only health and attack stats?"

In [ ]:
"What card is both an enemy and a companion card?"

In [ ]:
"What cards are can be both a player and enemy clunker?"

In [ ]:
"What fights can Gobling spawn in?"
"What fights can Gobling NOT spawn in?"

In [1]:
from neo4j import GraphDatabase
import os 
from dotenv import load_dotenv
load_dotenv(dotenv_path="configs/.env")

# Set up connection details
uri = "bolt://localhost:7687"  # update if using cloud or custom port
username = os.getenv('NEO4J_USERNAME')          # update with actual username
password = os.getenv('NEO4J_PASSWORD')          # update with actual password
driver = GraphDatabase.driver(uri, auth=(username, password))

query = """
MATCH (c:Card)
RETURN (c) AS card
"""

with driver.session() as session:
    result = session.run(query)
    card_dicts = [dict(record["card"]) for record in result]
        


In [2]:
card_dicts

[{'attack': 2,
  'card_url': 'https://wildfrostwiki.com/Binku',
  'card_name': 'Binku',
  'health': 5,
  'counter': 4,
  'card_type': 'pets',
  'card_description': 'Binku is a pet Companion, and the sixth available pet.'},
 {'attack': 3,
  'card_url': 'https://wildfrostwiki.com/Booshu',
  'card_name': 'Booshu',
  'health': 4,
  'counter': 5,
  'card_type': 'pets',
  'card_description': 'Booshu is a pet Companion, and the second available pet.'},
 {'attack': 4,
  'card_url': "https://wildfrostwiki.com/Lil'_Gazi",
  'card_name': "Lil' Gazi",
  'health': 3,
  'counter': 4,
  'card_type': 'pets',
  'card_description': "Lil' Gazi is a pet Companion, and the sixth and final pet."},
 {'attack': 3,
  'card_url': 'https://wildfrostwiki.com/Loki',
  'card_name': 'Loki',
  'health': 5,
  'counter': 3,
  'card_type': 'pets',
  'card_description': 'Loki is a pet Companion, and the third available pet.'},
 {'attack': 2,
  'card_url': 'https://wildfrostwiki.com/Sneezle',
  'card_name': 'Sneezle',
  '

In [15]:
def create_stats_queries(card_dicts, stats_query_template="In Wildfrost, what are the stats for {card}?"):
    keys_of_interest = ["health", "counter", "scrap", "attack"]
    queries = []

    for card in card_dicts:
        card_name = card.get("card_name", "Unknown Card")
        query_text = stats_query_template.format(card=card_name)
        
        # Extract stats if present
        ground_truth = {key: card[key] for key in keys_of_interest if key in card}

        if not ground_truth:
            ground_truth = "No stats for this card"
        else:
            # Add prefix text to the stats output
            stats_text = ", ".join(f"{key}: {value}" for key, value in ground_truth.items())
            ground_truth = "The stats are:\n" + stats_text

        queries.append({
            "query": query_text,
            "ground_truth": ground_truth
        })
    return queries


In [16]:
query_list = create_stats_queries(card_dicts)
for item in query_list:
    print(item)

{'query': 'In Wildfrost, what are the stats for Binku?', 'ground_truth': 'The stats are:\nhealth: 5, counter: 4, attack: 2'}
{'query': 'In Wildfrost, what are the stats for Booshu?', 'ground_truth': 'The stats are:\nhealth: 4, counter: 5, attack: 3'}
{'query': "In Wildfrost, what are the stats for Lil' Gazi?", 'ground_truth': 'The stats are:\nhealth: 3, counter: 4, attack: 4'}
{'query': 'In Wildfrost, what are the stats for Loki?', 'ground_truth': 'The stats are:\nhealth: 5, counter: 3, attack: 3'}
{'query': 'In Wildfrost, what are the stats for Sneezle?', 'ground_truth': 'The stats are:\nhealth: 6, counter: 3, attack: 2'}
{'query': 'In Wildfrost, what are the stats for Snoof?', 'ground_truth': 'The stats are:\nhealth: 3, counter: 3, attack: 3'}
{'query': 'In Wildfrost, what are the stats for Spike?', 'ground_truth': 'The stats are:\nhealth: 7'}
{'query': 'In Wildfrost, what are the stats for Alloy?', 'ground_truth': 'The stats are:\nhealth: 12, counter: 5, attack: 6'}
{'query': 'In Wi

In [17]:
import asyncio
import json
from ollama import AsyncClient

async def query_single(card_query, model_name="gemma3", client=None):
    query_text = card_query["query"]
    ground_truth = card_query["ground_truth"]
    system_prompt = (
        "You are a helpful assistant that only uses the provided information to answer queries.\n"
        "Answer based only on this data. Ensure you provide a full answer to the query."
    )

    ground_truth_str = json.dumps(ground_truth)

    response = await client.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query_text + "\n" + ground_truth_str},
        ],
    )
    print(f"Query: {query_text}")
    print(f"Response: {response['message']['content']}")
    print("-" * 40)

async def query_ollama_model_async(card_queries, model_name="gemma3"):
    client = AsyncClient()
    tasks = [
        query_single(card_query=item, model_name=model_name, client=client)
        for item in card_queries
    ]
    await asyncio.gather(*tasks)


In [18]:
# Example usage:
card_queries = create_stats_queries(card_dicts)


In [19]:
len(card_queries)

283

In [20]:
await query_ollama_model_async(card_queries)

Query: In Wildfrost, what are the stats for Groff?
Response: The stats for Groff are: health: 6, attack: 5.
----------------------------------------
Query: In Wildfrost, what are the stats for Loki?
Response: Loki's stats are: health: 5, counter: 3, attack: 3.
----------------------------------------
Query: In Wildfrost, what are the stats for Blunky?
Response: The stats for Blunky are: health: 1, counter: 2, attack: 1.
----------------------------------------
Query: In Wildfrost, what are the stats for Berry Sis?
Response: The stats for Berry Sis are: health: 8, counter: 3, attack: 2.
----------------------------------------
Query: In Wildfrost, what are the stats for Splinter?
Response: The stats for Splinter are: health: 4, counter: 4, attack: 4.
----------------------------------------
Query: In Wildfrost, what are the stats for Van Jun?
Response: Van Jun's stats are: health: 4, counter: 4.
----------------------------------------
Query: In Wildfrost, what are the stats for Tiny Ty